In [1]:
import sys

sys.path.append("../src/legifrance")

In [2]:
from legifrance_api_client import LegiFranceClient
client = LegiFranceClient()

In [4]:
payload = {
  "recherche": {
    "filtres": [
      {
        "dates": {
          "start": "2017-09-01",
          "end": "2025-12-31"
        },
        "facette": "DATE_SIGNATURE"
      },
      {
        "dates": {
          "start": "2026-09-14",
          "end": "2026-09-18"
        },
        "facette": "DATE_DIFFUSION"
      },
    ],
    "sort": "DATE_ASC",
    "secondSort": "ID",
    "fromAdvancedRecherche": False,
    "pageSize": 100,
    "typePagination": "DEFAUT",
    "pageNumber": 1
  },
  "fond": "ACCO"
}
totalResultNumber = client.search(payload=payload).json()["totalResultNumber"]

In [5]:
import numpy as np

In [11]:
maxPages = int(np.ceil(totalResultNumber/100))
new_acco_cids = []
new_acco_dates = []

for page in range(1, maxPages + 1):
    payload["recherche"]["pageNumber"] = page
    results = client.search(payload=payload).json()["results"]
    for result in results:
        cid = result["titles"][0]["cid"]
        new_acco_cids.append(cid)

        date = result["dateSignature"][:7].replace("-", "/")
        new_acco_dates.append(date)

In [15]:
payload = {"id": new_acco_cids[0]}

response = client.download_acco(payload)

In [19]:
response.json()["acco"]["data"]

'...'

In [21]:
import base64
import s3fs

fs = s3fs.S3FileSystem(
    endpoint_url="https://minio.lab.sspcloud.fr",
    client_kwargs={"region_name": "us-east-1"},
)

DOCUMENTS_PATH = "s3://mateomorin/legifrance/documents/"

for cid, date in zip(new_acco_cids, new_acco_dates):
    path_to_upload = DOCUMENTS_PATH + date + cid + ".docx"
    payload = {"id": cid}

    content_b64 = client.download_acco(payload).json()["acco"]["data"]
    binary_data = base64.b64decode(content_b64)

    with fs.open(path_to_upload, "wb") as f:
        f.write(binary_data)


In [27]:
new_acco_cids[-1]

'ACCOTEXT000054835696'

In [28]:
new_acco_dates[-1]

'2025/12'

In [29]:
fs.find(DOCUMENTS_PATH + "2025/12/ACCOTEXT000054835696.docx")

['mateomorin/legifrance/documents/2025/12/ACCOTEXT000054835696.docx']

In [33]:
import pandas as pd

df = pd.read_parquet("s3://mateomorin/legifrance/metadata/acco_metadata_2024.parquet", filesystem=fs)

df["first_ref"] = df["reference"].str[0]

df.value_counts("first_ref")

first_ref
T    45541
A      151
Name: count, dtype: int64